# TIGER on Amazon Toys & Games — Colab driver

**Group 15 — Reza, Lara, Nithin.** RS course 2025/26, Assignment 3.

Self-driving notebook: it clones the implementation from GitHub, installs requirements, mounts Drive (optional, for resume-across-sessions), and runs the three TIGER stages end-to-end.

**Runtime → Change runtime type → GPU → T4** before you start. Then `Runtime → Run all`.

Sections:
1. Verify the T4 GPU.
2. Clone the repo from GitHub, install deps.
3. (Optional) Mount Google Drive and reuse cached embeddings / checkpoints across sessions.
4. Stage 0 — Sentence-T5 content embeddings.
5. Stage 1 — RQ-VAE training + Semantic ID extraction.
6. Stage 2 — Generative Transformer training + final eval.
7. Ablations (optional — uncomment to run).

## 1. GPU check

In [ ]:
!nvidia-smi

## 2. Clone repo + install requirements

In [ ]:
!git clone --depth 1 https://github.com/nithin-06/recommender_systems_assignment.git /content/repo
%cd /content/repo/Assignment 3
!pip install -q -r requirements.txt

## 3. (Optional) Mount Drive for persistence

Set `USE_DRIVE = True` if you've created `MyDrive/tiger_assignment/{data,embeddings,checkpoints}` and want training to checkpoint there. Otherwise everything goes into `/content` and is lost on session end.

Either way, the dataset files are downloaded automatically — you do **not** need to manually upload anything.

In [ ]:
USE_DRIVE = False    # ← set to True once your Drive folders exist

import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = '/content/drive/MyDrive/tiger_assignment'
    os.environ['TIGER_DATA_DIR']    = f'{ROOT}/data'
    os.environ['TIGER_EMB_DIR']     = f'{ROOT}/embeddings'
    os.environ['TIGER_CKPT_DIR']    = f'{ROOT}/checkpoints'
    os.environ['TIGER_RESULTS_DIR'] = f'{ROOT}/results'
    !mkdir -p '{ROOT}/data' '{ROOT}/embeddings' '{ROOT}/checkpoints' '{ROOT}/results'
else:
    os.environ['TIGER_DATA_DIR']    = '/content/repo/Assignment 3/data'
    os.environ['TIGER_EMB_DIR']     = '/content/repo/Assignment 3/data/embeddings'
    os.environ['TIGER_CKPT_DIR']    = '/content/repo/Assignment 3/data/checkpoints'
    os.environ['TIGER_RESULTS_DIR'] = '/content/repo/Assignment 3/results'

# Auto-download dataset if missing
import urllib.request, pathlib
DATA_DIR = pathlib.Path(os.environ['TIGER_DATA_DIR'])
DATA_DIR.mkdir(parents=True, exist_ok=True)
for fname in ('reviews_Toys_and_Games_5.json.gz', 'meta_Toys_and_Games.json.gz'):
    path = DATA_DIR / fname
    if not path.exists():
        url = f'http://snap.stanford.edu/data/amazon/productGraph/categoryFiles/{fname}'
        print(f'downloading {fname} ...')
        urllib.request.urlretrieve(url, path)
    print(f'{fname}: {path.stat().st_size/1e6:.1f} MB')

## 4. Stage 0 — Sentence-T5 content embeddings

On a T4 this takes ~3 min. The output is cached, so re-running this cell is instant.

In [ ]:
!python main.py --stage embed --tag embed

## 5. Stage 1 — RQ-VAE

Trains the residual quantizer (L=3, K=256, 32-dim latent) on the cached content embeddings. Reports codebook usage and collision rate.

In [ ]:
!python main.py --stage rqvae --tag rqvae_default \
    --rq_levels 3 --rq_codebook_size 256 --rq_latent_dim 32 --rq_epochs 200

## 6. Stage 2 — Generative Transformer

Paper defaults: 4 enc + 4 dec layers, 6 heads × 64 = dim 384, FFN 1024, dropout 0.1, beam 10 at inference. Trains until validation NDCG@10 plateaus.

In [ ]:
!python main.py --stage tiger --tag tiger_default --tiger_tag tiger_default \
    --t_hidden 384 --t_heads 6 --t_layers 4 --t_ffn 1024 --t_dropout 0.1 \
    --t_lr 1e-3 --t_batch_size 256 --t_epochs 200 --beam_size 10

## 7. Ablations (uncomment + run any you want)

Each ablation reuses the cached content embeddings; only RQ-VAE ablations retrain the quantizer.

In [ ]:
# RQ-VAE codebook size sweep -- need separate quantizers per K
# !python main.py --stage rqvae --tag rqvae_K64  --rqvae_tag rqvae_K64  --rq_codebook_size 64
# !python main.py --stage rqvae --tag rqvae_K128 --rqvae_tag rqvae_K128 --rq_codebook_size 128

# RQ-VAE levels sweep
# !python main.py --stage rqvae --tag rqvae_L2 --rqvae_tag rqvae_L2 --rq_levels 2
# !python main.py --stage rqvae --tag rqvae_L4 --rqvae_tag rqvae_L4 --rq_levels 4

# Transformer width / depth / heads ablations -- reuse default RQ-VAE
# !python main.py --stage tiger --tag tiger_small  --tiger_tag tiger_small  --t_hidden 192 --t_heads 3
# !python main.py --stage tiger --tag tiger_large  --tiger_tag tiger_large  --t_hidden 768 --t_heads 12
# !python main.py --stage tiger --tag tiger_deep   --tiger_tag tiger_deep   --t_layers 6
# !python main.py --stage tiger --tag tiger_shallow --tiger_tag tiger_shallow --t_layers 2

# Beam size ablation -- reuse trained Transformer; rebuilds eval only
# !python main.py --stage tiger --tag tiger_beam5  --tiger_tag tiger_default --beam_size 5
# !python main.py --stage tiger --tag tiger_beam20 --tiger_tag tiger_default --beam_size 20

## 8. Inspect results

In [ ]:
import json, pathlib
for p in sorted(pathlib.Path(os.environ['TIGER_RESULTS_DIR']).glob('*.json')):
    payload = json.loads(p.read_text())
    test = payload.get('test', {})
    coll = payload.get('collision_stats', {})
    print(f"{p.stem:24s}  recall@10={test.get('recall@10', float('nan')):.4f}  ndcg@10={test.get('ndcg@10', float('nan')):.4f}  invalid={test.get('invalid_rate', float('nan')):.3f}  collisions={coll.get('collision_rate', float('nan')):.3f}")